# Process Documents

In [ ]:
import os

import pandas as pd
from docling.document_converter import DocumentConverter

from agent_k.config.logger import logger

# Files that are taking too long to process. Exclude for now and process later if needed.
files_to_exclude = [
    "02711c830d14cbb8e365574b7aa58d792209a84d07eabf7c7354c342c93e260991.pdf"
]

df_inferlink_gt = pd.read_csv("data/processed/ground_truth/inferlink_ground_truth.csv")
record_ids = df_inferlink_gt["cdr_record_id"].tolist()

# Search each record ID in the 43-101 directory "data/raw/all_sources/43-101"
pdf_paths = []
for record_id in record_ids:
    for file in os.listdir("data/raw/all_sources/43-101"):
        if file in files_to_exclude or ".dvc" in file:
            logger.info(
                f"Skipping because it is in the exclude list or a dvc file: {file}"
            )
            continue
        if record_id in file:
            pdf_paths.append(os.path.join("data/raw/all_sources/43-101", file))

converter = DocumentConverter()

for i, pdf_path in enumerate(pdf_paths):
    logger.info(f"{i + 1}/{len(pdf_paths)}: Processing {pdf_path}")

    # Check if the markdown file already exists
    record_id = pdf_path.split("/")[-1].split(".")[0]
    if os.path.exists(os.path.join("data/processed/43-101", f"{record_id}.md")):
        logger.info("Skipping because it already exists")
        continue

    result = converter.convert(pdf_path)
    # Save the markdown to a file in processed/43-101
    with open(os.path.join("data/processed/43-101", f"{record_id}.md"), "w") as f:
        f.write(result.document.export_to_markdown())

## Refine Headers
1. Removing false positive headers in markdown splitters (e.g. "## Notes:")
2. Correct header levels

In [ ]:
import os
import re
from pathlib import Path


def extract_headers_from_markdown(markdown_text):
    """
    Extract all headers from a markdown text.

    Args:
        markdown_text (str): The markdown text to parse

    Returns:
        list: A list of tuples containing (header_level, header_text)
    """
    # Regular expression to match markdown headers (# Header)
    header_pattern = re.compile(r"^(#{1,6})\s+(.+?)(?:\s+#+)?$", re.MULTILINE)

    # Find all headers in the text
    headers = []
    for match in header_pattern.finditer(markdown_text):
        level = len(match.group(1))  # Number of # characters
        text = match.group(2).strip()
        # If header text begins with "1.0", "2.0", "3.0", "<int>.0", etc., replace with "1", "2", "3", "<int>", etc.
        if re.match(r"^\d+\.0", text):
            print(f"Removing .0 from {text}")
            text = re.sub(r"\.0", "", text)
            print(f"New text: {text}")
        headers.append((level, text))

    return headers


# Example usage
markdown_dir = Path("data/processed/43-101")
sample_files = list(markdown_dir.glob("*.md"))[
    4:7
]  # Get first 3 files for demonstration

for file_path in sample_files:
    print(f"\nHeaders in {file_path.name}:")

    with open(file_path, "r", encoding="utf-8") as f:
        markdown_content = f.read()

    headers = extract_headers_from_markdown(markdown_content)

    for level, text in headers:
        print(f"{'  ' * (level - 1)}{'#' * level} {text}")

In [ ]:
def correct_header_levels(headers):
    """
    Correct header levels in markdown headers based on specified heuristics:
    1. By default keep header level as level 2
    2. Numbered headers get levels based on their numbering depth relative to the highest level:
       - If highest level is "1", then "1" -> level 2, "1.1" -> level 3, etc.
       - If highest level is "1.0", then "1.0" -> level 2, "1.0.1" -> level 3, etc.
    3. Unnumbered headers between numbered headers get level one below the enclosing numbered headers
    4. Replace "## Notes:", "## Notes", or "## Note" with "Notes:"

    Args:
        headers: List of tuples (header_level, header_text)

    Returns:
        List of tuples with corrected (header_level, header_text)
    """
    if not headers:
        return []

    import re

    # First pass: Determine the highest level numbered header pattern and count how many numbered headers there are
    highest_level_parts = None
    numbered_headers_info = []
    numbered_headers_count = 0

    for i, (_, text) in enumerate(headers):
        number_match = re.match(r"^(\d+(\.\d+)*)(\s+|\.|\s*$)", text)
        if number_match:
            numbered_headers_count += 1
            number = number_match.group(1)
            parts = number.split(".")

            # Store information about this numbered header
            numbered_headers_info.append((i, number, parts))

            # Update highest_level_parts if this is a higher level
            if highest_level_parts is None or len(parts) < len(highest_level_parts):
                highest_level_parts = parts

    # If no numbered headers found, highest level is None
    base_level_length = len(highest_level_parts) if highest_level_parts else 0

    # Second pass: Process all headers with the determined base level
    corrected_headers = []
    for i in range(len(headers)):
        original_level, text = headers[i]

        # If header text smaller than 3 characters, set the header level to -1 to remove it
        if len(text) < 3:
            corrected_headers.append((-1, text))
            continue

        # Check if this is a numbered header
        number_match = re.match(r"^(\d+(\.\d+)*)(\s+|\.|\s*$)", text)
        if number_match:
            # This is a numbered header
            number = number_match.group(1)
            parts = number.split(".")

            # Calculate level relative to the highest level
            # The highest level becomes level 2
            level_difference = len(parts) - base_level_length
            level = 2 + level_difference

            corrected_headers.append((level, text))
        else:
            # This is an unnumbered header

            # Check if there's a numbered header preceding this one
            has_preceding_numbered = False
            prev_number_header_level = None

            # Look back through previous headers to find the most recent numbered header
            for j in range(i - 1, -1, -1):
                _, prev_text = headers[j]
                prev_number_match = re.match(r"^(\d+(\.\d+)*)(\s+|\.|\s*$)", prev_text)
                if prev_number_match:
                    has_preceding_numbered = True
                    prev_number_header_level = corrected_headers[j][0]
                    break

            if has_preceding_numbered and numbered_headers_count > 3:
                corrected_headers.append((prev_number_header_level + 1, text))
            else:
                # Default to level 2 if no preceding numbered header
                corrected_headers.append((2, text))

    return corrected_headers


def preprocess_markdown(markdown_content):
    """
    Replace false positive headers in markdown (e.g. "## Notes:") with normal content "Notes:"
    """

    # Replace some false positive headers with normal content
    notes_header_regex = re.compile(
        r"^(#{1,6})\s?note[s]?", re.IGNORECASE | re.MULTILINE
    )
    markdown_content = re.sub(notes_header_regex, "Notes:", markdown_content)

    notes_header_regex = re.compile(
        r"^(#{1,6})\s?figure[s]?", re.IGNORECASE | re.MULTILINE
    )
    markdown_content = re.sub(notes_header_regex, "Figure:", markdown_content)

    notes_header_regex = re.compile(
        r"^(#{1,6})\s?table[s]?", re.IGNORECASE | re.MULTILINE
    )
    markdown_content = re.sub(notes_header_regex, "Table:", markdown_content)

    return markdown_content


if __name__ == "__main__":
    # markdown_dir = Path("data/processed/43-101")
    # sample_files = list(markdown_dir.glob("*.md"))[4]  # Get first file
    sample_file_path = "data/processed/43-101/022b81881794b6910528a035b50a214fc960c52f89d7f84a35ce1b75b96f3759f0.md"
    with open(sample_file_path, "r", encoding="utf-8") as f:
        markdown_content = f.read()
    markdown_content = preprocess_markdown(markdown_content)
    headers = extract_headers_from_markdown(markdown_content)

    print("\nOriginal headers:")
    for level, text in headers:
        print(f"{'  ' * (level - 1)}{'#' * level} {text}")

    corrected_headers = correct_header_levels(headers)

    print("\nCorrected headers:")
    for level, text in corrected_headers:
        if level != -1:
            print(f"{'  ' * (level - 1)}{'#' * level} {text}")
        else:
            print(f"===========\nSkipping {text}\n===========")

In [ ]:
# Write a function to correct and replace the headers in all markdown files in the data/processed/43-101 directory and write the corrected markdown to a new folder in the data/processed/43-101-refined directory
import os


def correct_headers_in_directory(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for file_path in Path(input_dir).glob("*.md"):
        with open(file_path, "r", encoding="utf-8") as f:
            markdown_content = f.read()
        markdown_content = preprocess_markdown(markdown_content)
        headers = extract_headers_from_markdown(markdown_content)
        corrected_headers = correct_header_levels(headers)
        for old_header, new_header in zip(headers, corrected_headers, strict=False):
            if new_header[0] != -1:
                markdown_content = markdown_content.replace(
                    f"{'#' * old_header[0]} {old_header[1]}",
                    f"{'#' * new_header[0]} {new_header[1]}",
                )
            else:
                markdown_content = markdown_content.replace(
                    f"{'#' * old_header[0]} {old_header[1]}",
                    f"{old_header[1]}",
                )
        with open(
            os.path.join(output_dir, file_path.name),
            "w",
            encoding="utf-8",
        ) as f:
            f.write(markdown_content)


correct_headers_in_directory(
    input_dir="data/processed/43-101",
    output_dir="data/processed/43-101-refined",
)